# Calculate 2D electric field with $dr d\Phi$

In [11]:
from matplotlib import pyplot as plt
import numpy as np
from scipy.special import jv, kv
from scipy.special import j1 # Bessel function of the first kind of order 1, for the diffraction pattern calculation
from scipy.optimize import root_scalar
from scipy.constants import c, epsilon_0, mu_0, pi
import scipy.integrate as integrate

In [12]:
# jn_zeros is a function that returns the zeros of the Bessel function of the first kind, 
# which are needed to find the roots for the LP01 mode in a step-index fibre
from scipy.special import jn_zeros  # NEW: needed to get the first zero of J0

In [13]:
from scipy.optimize import brentq
from scipy.optimize import fminbound
# The Brent's method is a root-finding algorithm that combines the bisection method, the secant method, and 
# inverse quadratic interpolation. 
# https://en.wikipedia.org/wiki/Brent%27s_method
# It is designed to find roots of a continuous function within a specified interval where the function changes
#  sign. The method is efficient and robust, making it suitable for finding roots of nonlinear equations.

In [14]:
from dataclasses import dataclass

In [ ]:
def cutoff(l,m):
    """
    Utility  function which returns the normalised frequency (V) at cutoff V for
    a given mode l,m. 
    
    Arguments: 
    l: Azimuthal order of the LP mode, integer >= 0
    m: Radial order of the LP mode, integer >= 1
    
    Return value: 

    Vc: Normalised frequency at cutoff for teh given mode, equals to the last zero of J_{l-1} for l>=1, 
    and Vc=0 for J_{m-1} for l=0.
    
    """

    if l >= 1: 
        return jn_zeros(l - 1, m)[-1]
    elif l == 0: 
        if m == 1: 
            return 0.0 
        else:
            return jn_zeros(1, m - 1)[-1]
    else:
        raise ValueError("l must be >= 0")

In [17]:
def modes(V, l_max=5, m_max=5, v_margin=2e-3, az_sym_only=True): 
    """
    For each value of l, calculate the allowed values of m (if any). 

    Arguments: 
    V: Normalised frequency of the fiber, float > 0
    l_max: Maximum azimuthal order to consider, integer >= 0
    m_max: Maximum radial order to consider, integer >= 1
    v_margin: Margin for checking mode existence at cutoff, to account for numerical precision issues. 
    az_sym_only: If True, only include modes with l=0 (azimuthally symmetric modes). If False, include all modes up to l_max.
    Return value: 
    mode_list: A list of [l, m, Cv] for each mode that is supported by the fibre, where Vc is the cutoff V-number for that mode. 
    """ 
    if V <= 0: 
        raise ValueError("V must be >= 0")

    mode_list = []

    for l in range(l_max + 1): 
        if az_sym_only and l != 0: 
            continue

        for m in range(1, m_max + 1): 
            V_c = cutoff(l, m) 

            if V > V_c + v_margin: 
                mode_list.append([l, m, V_c])

            else: 
                break
    
    # A lambda function is an anonymous function in Python, which can take any number of arguments but can only have one expression. 
    # In this case, the lambda function is used as the key for sorting the mode_list. 
    mode_list.sort(key=lambda mode: mode[2])  # Sort by cutoff V-number

    return mode_list

In [ ]:
def function_characteristic_eq(X, V, l): 
    """
    Function that equates to zero at the roots of the characteristic equation. 

    Arguments: 
    X: Normalised transverse propagation constant in the core, float > 0
    V: Normalised frequency of the fibre, float > 0
    Return value: 
    lhs - rhs: The difference between the left-hand side and right-hand side of the characteristic equation, which should be zero at the roots. 
    """
    # Calculate Y based on the relationship Y^2 = V^2 - X^2
    Y = np.sqrt(V**2 - X**2) # Y is the normalised transverse propagation constant in the cladding, and must be real for guided modes (X < V)

    if l > 0: 
        lhs = X * jv(l - 1, X) / jv(l, X)
        rhs = Y * kv(l - 1, Y) / kv(l, Y)
        return lhs + rhs 
    
    elif l == 0:
        lhs = X * jv(1, X) / jv(0, X)
        rhs = Y * kv(1, Y) / kv(0, Y)
        return lhs - rhs
    
    else:
        raise ValueError("l must be >= 0")

In [3]:
def function_LP01(X, V):
    ""
    # Calculate Y based on the relationship Y^2 = V^2 - X^2
    Y = np.sqrt(V**2 - X**2)
    # Calculate the left-hand side and right-hand side of the equation using the Bessel functions
    lhs = X * jv(1, X) / jv(0, X)
    rhs = Y * kv(1, Y) / kv(0, Y)
    return lhs - rhs

def find_root(V, X_min, X_max):
    # Define a lambda function that takes X as input and uses the previously defined function f with the 
    # fixed V value. This allows us to use this function for root finding.
    function_m1 = lambda X: function_LP01(X, V)

    X_scan = np.linspace(X_min, X_max, 5000)
    F_scan = function_m1(X_scan) # Evaluate the function at the scan points

    bracket = None

    # We take length of scan - 1 because we are checking pairs of points for sign changes, 
    # -1 means we won't go out of bounds when checking F_scan[i + 1]
    for i in range(len(X_scan) - 1): # Loop through the length-1 of X_scan to find a sign change
        if F_scan[i] * F_scan[i + 1] < 0: # Check for a sign change
            x1 = X_scan[i]
            x2 = X_scan[i + 1]
            f1 = F_scan[i]
            f2 = F_scan[i + 1]

            # Skip non-finite values
            if not np.isfinite(f1) or not np.isfinite(f2):
                continue

            # sign chage detected between x1 and x2, we can use this as a bracket for root finding
            if f1*f2 < 0:
                bracket = (x1, x2) # Store the bracket where the sign change occurs
                break # Exit the loop after finding the first sign change

    if bracket is None:
        raise ValueError(f"No valid bracket found for V = {V:.6f}")

    X_root = brentq(function_m1, bracket[0], bracket[1]) # Use the found bracket for root finding
    Y_root = np.sqrt(V**2 - X_root**2)
    return X_root, Y_root


## 2D E-field of misaligned fibre

Start considering the misalignment of the E-field because we can always have a deviation in alignment between the fibre and the pupil. 

In [ ]:
def calculate_electric_field_2d(r, a, phi, X_root, Y_root):
    
    # Create a 2D grid in physical coordinates
    x = np.linspace(-3, 3, 1000) * a
    y = np.linspace(-3, 3, 1000) * a
    X_grid, Y_grid = np.meshgrid(x, y)

    X_shifted = X_grid - a * np.cos(phi) # Shift the grid in x direction by a*cos(phi)
    Y_shifted = Y_grid - a * np.sin(phi) # Shift the grid in y direction by a*sin(phi)

    rho_2d = np.sqrt(X_grid**2 + Y_grid**2) # Calculate the radial distance from the center of the fiber
    phi_2d = np.arctan2(Y_grid, X_grid) # Calculate the angular coordinate in the 2D grid

    # Calculate the electric field distribution for the misaligned case
    # Empty 2D field array
    E_2d = np.zeros_like(rho_2d)

    # Core and cladding masks
    core = rho_2d <= 1
    cladding = rho_2d > 1

    # Core field
    E_2d[core] = jv(0, X_root * rho_2d[core])

    # misaligned field in the core (using the same Bessel function but with a phase shift)
    E_2d[core] *= np.exp(1j * phi) # Apply the phase shift to the core field to represent misalignment

    # Matching factor for continuity at rho = 1
    match_factor = jv(0, X_root) / kv(0, Y_root)

    # Cladding field
    E_2d[cladding] = match_factor * kv(0, Y_root * rho_2d[cladding])

    # Normalize field if desired
    E_2d_norm = E_2d / np.max(np.abs(E_2d))

    # Intensity
    I_2d = np.abs(E_2d_norm)**2

    return E_2d_norm, I_2d